In [1]:
import os

# Tell Python where our training data is located
train_dir = "data/Training"

# Ask Python to count the images in each folder
for folder in os.listdir(train_dir):
    folder_path = os.path.join(train_dir, folder)
    num_images = len(os.listdir(folder_path))
    print(f"The {folder} folder has {num_images} images.")

The glioma folder has 1400 images.
The meningioma folder has 1400 images.
The notumor folder has 1400 images.
The pituitary folder has 1400 images.


In [9]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# 1. The NEW Training Preprocessor (With Data Augmentation!)
train_data_gen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,      # Randomly rotate images up to 15 degrees
    width_shift_range=0.1,  # Randomly shift images left/right
    height_shift_range=0.1, # Randomly shift images up/down
    zoom_range=0.1,         # Randomly zoom in or out slightly
    horizontal_flip=True    # Randomly flip images like a mirror
)

# 2. The Testing Preprocessor (NO scrambling here, just shrinking the math)
test_data_gen = ImageDataGenerator(rescale=1./255)

# 3. The Training Conveyor Belt
print("Loading Training Data:")
train_data = train_data_gen.flow_from_directory(
    'data/Training',
    target_size=(150, 150),
    batch_size=32,
    class_mode='categorical'
)

# 4. The Testing Conveyor Belt
print("\nLoading Testing Data:")
test_data = test_data_gen.flow_from_directory(
    'data/Testing',
    target_size=(150, 150),
    batch_size=32,
    class_mode='categorical'
)

Loading Training Data:
Found 5600 images belonging to 4 classes.

Loading Testing Data:
Found 1600 images belonging to 4 classes.


In [5]:
from tensorflow.keras import layers, models

# 1. Initialize the Brain (A blank slate)
model = models.Sequential()

# 2. First Magnifying Glass (Looking for basic edges)
# We use 32 filters (glasses), each 3x3 pixels wide.
model.add(layers.Conv2D(32, (3, 3), activation='relu', input_shape=(150, 150, 3)))
model.add(layers.MaxPooling2D((2, 2))) # This shrinks the image to focus only on the loudest features

# 3. Second Magnifying Glass (Looking for shapes)
model.add(layers.Conv2D(64, (3, 3), activation='relu'))
model.add(layers.MaxPooling2D((2, 2)))

# 4. Third Magnifying Glass (Looking for complex tumor textures)
model.add(layers.Conv2D(128, (3, 3), activation='relu'))
model.add(layers.MaxPooling2D((2, 2)))

# 5. The Translator 
# The AI has been looking at 2D grids. We need to flatten that grid into a single list of numbers.
model.add(layers.Flatten())

# 6. The Decision Makers
model.add(layers.Dense(128, activation='relu'))       # The neurons that weigh the evidence
model.add(layers.Dense(4, activation='softmax'))      # The final output: 4 probabilities (1 for each tumor type)

# 7. Give the Brain its Instructions
model.compile(optimizer='adam',                       # The algorithm that updates the math as it learns
              loss='categorical_crossentropy',        # How the AI calculates how "wrong" its guesses are
              metrics=['accuracy'])                   # We want to measure success based on accuracy

# Print out the blueprint of our brain
model.summary()

C:\Users\divya\Desktop\MINOR_PROJECT_2\venv\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                      │ (None, 148, 148, 32)        │             896 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d (MaxPooling2D)         │ (None, 74, 74, 32)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_1 (Conv2D)                    │ (None, 72, 72, 64)          │          18,496 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_1 (MaxPooling2D)       │ (None, 36, 36, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_2 (Conv2D)                    │ (None, 34, 34, 128)         │          73,856 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_2 (MaxPooling2D)       │ (None, 17, 17, 128)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten (Flatten)                    │ (None, 36992)               │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 128)                 │       4,735,104 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 4)                   │             516 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 4,828,868 (18.42 MB)

 Trainable params: 4,828,868 (18.42 MB)

 Non-trainable params: 0 (0.00 B)

In [11]:
# 8. Train the Brain (This will take a few minutes!)
history = model.fit(
    train_data,
    epochs=10,                  # How many times the AI will read the entire textbook
    validation_data=test_data   # The exam it takes after every read-through
)

# 9. Save the Brain to a file so we can use it in our web app later!
model.save('tumor_classifier_model.h5')
print("Model successfully saved!")

Epoch 1/10
175/175 ━━━━━━━━━━━━━━━━━━━━ 143s 811ms/step - accuracy: 0.7507 - loss: 0.6576 - val_accuracy: 0.7944 - val_loss: 0.8003
Epoch 2/10
175/175 ━━━━━━━━━━━━━━━━━━━━ 129s 738ms/step - accuracy: 0.8254 - loss: 0.4429 - val_accuracy: 0.8050 - val_loss: 0.7446
Epoch 3/10
175/175 ━━━━━━━━━━━━━━━━━━━━ 182s 1s/step - accuracy: 0.8441 - loss: 0.4051 - val_accuracy: 0.8313 - val_loss: 0.7394
Epoch 4/10
175/175 ━━━━━━━━━━━━━━━━━━━━ 126s 717ms/step - accuracy: 0.8675 - loss: 0.3409 - val_accuracy: 0.8506 - val_loss: 0.7537
Epoch 5/10
175/175 ━━━━━━━━━━━━━━━━━━━━ 181s 1s/step - accuracy: 0.8841 - loss: 0.3100 - val_accuracy: 0.8338 - val_loss: 0.7695
Epoch 6/10
175/175 ━━━━━━━━━━━━━━━━━━━━ 165s 938ms/step - accuracy: 0.8941 - loss: 0.2749 - val_accuracy: 0.8619 - val_loss: 0.8154
Epoch 7/10
175/175 ━━━━━━━━━━━━━━━━━━━━ 179s 1s/step - accuracy: 0.9055 - loss: 0.2555 - val_accuracy: 0.8763 - val_loss: 0.7590
Epoch 8/10
175/175 ━━━━━━━━━━━━━━━━━━━━ 160s 917ms/step - accuracy: 0.9038 - loss: 0.

Model successfully saved!


In [7]:
print(train_data.class_indices)

{'glioma': 0, 'meningioma': 1, 'notumor': 2, 'pituitary': 3}
